In [ ]:
#Cheap items: many
#Expensive items: few but influential

In [ ]:
!pip install transformers scikit-learn


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression


In [ ]:
df = pd.read_csv("/content/Transformer_train.csv")

df = df.rename(columns={
    "catalog_content": "text",
    "price": "price"
})



In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)


In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
print(train_df.columns)
print(val_df.columns)


Index(['sample_id', 'text', 'price'], dtype='object')
Index(['sample_id', 'text', 'price'], dtype='object')


In [ ]:
import numpy as np

# Create log-transformed target
df["log_price"] = np.log1p(df["price"])

# Re-split AFTER adding log_price
from sklearn.model_selection import train_test_split
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)


In [ ]:
print(train_df.columns)


Index(['sample_id', 'text', 'price', 'log_price'], dtype='object')


In [ ]:
import torch
from torch.utils.data import Dataset

class PriceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=128):
        # Defensive checks
        assert "text" in df.columns, f"Missing 'text' column. Found: {df.columns}"
        assert "log_price" in df.columns, f"Missing 'log_price' column. Found: {df.columns}"

        self.texts = df["text"].astype(str).tolist()
        self.labels = df["log_price"].astype(float).values
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float)
        }



In [ ]:
class PricePredictor(nn.Module):
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        hidden = encoder.config.hidden_size

        self.regressor = nn.Sequential(
            nn.Linear(hidden, hidden // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden // 2, 1)
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # CLS pooling
        cls_embedding = outputs.last_hidden_state[:, 0, :]
        return self.regressor(cls_embedding).squeeze(-1)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = AutoModel.from_pretrained(MODEL_NAME)
model = PricePredictor(encoder).to(device)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [ ]:
def weighted_log_mse(pred, target):
    weight = torch.exp(-target)   # emphasize cheaper items
    return (weight * (pred - target) ** 2).mean()


In [ ]:
from torch.utils.data import DataLoader

train_dataset = PriceDataset(train_df, tokenizer)
val_dataset   = PriceDataset(val_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16)



In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

num_epochs = 20
total_steps = len(train_loader) * num_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)


In [ ]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        preds = model(input_ids, attention_mask)
        loss = weighted_log_mse(preds, labels)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_loss:.4f}")


Epoch 1/20 | Train Loss: 0.2959
Epoch 2/20 | Train Loss: 0.0582
Epoch 3/20 | Train Loss: 0.0446
Epoch 4/20 | Train Loss: 0.0336
Epoch 5/20 | Train Loss: 0.0255
Epoch 6/20 | Train Loss: 0.0193
Epoch 7/20 | Train Loss: 0.0153
Epoch 8/20 | Train Loss: 0.0155
Epoch 9/20 | Train Loss: 0.0111
Epoch 10/20 | Train Loss: 0.0094
Epoch 11/20 | Train Loss: 0.0074
Epoch 12/20 | Train Loss: 0.0077
Epoch 13/20 | Train Loss: 0.0071
Epoch 14/20 | Train Loss: 0.0061
Epoch 15/20 | Train Loss: 0.0053
Epoch 16/20 | Train Loss: 0.0055
Epoch 17/20 | Train Loss: 0.0053
Epoch 18/20 | Train Loss: 0.0048
Epoch 19/20 | Train Loss: 0.0048
Epoch 20/20 | Train Loss: 0.0047


In [ ]:
model.eval()
preds, trues = [], []

with torch.no_grad():
    for batch in val_loader:
        p = model(
            batch["input_ids"].to(device),
            batch["attention_mask"].to(device)
        ).cpu().numpy()
        preds.append(p)
        trues.append(batch["labels"].numpy())

preds = np.concatenate(preds)
trues = np.concatenate(trues)

calibrator = LinearRegression()
calibrator.fit(preds.reshape(-1, 1), trues)


LinearRegression()

In [ ]:
def predict_price(text):
    model.eval()

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding="max_length",
        max_length=128
    ).to(device)

    with torch.no_grad():
        log_price = model(
            inputs["input_ids"],
            inputs["attention_mask"]
        ).item()

    return np.expm1(log_price)


In [ ]:
def predict_price_calibrated(text):
    raw_price = predict_price(text)
    log_raw = np.log1p(raw_price)
    calibrated_log = calibrator.predict([[log_raw]])[0]
    return np.expm1(calibrated_log)


In [ ]:
sample_text = "Item Name: Deep Authentic Indian Spice Mix Blend"

price = predict_price_calibrated(sample_text)

print(f"Target Description: {sample_text}")
print(f"Optimized Predicted Price: ${price:.2f}")


Target Description: Item Name: Deep Authentic Indian Spice Mix Blend
Optimized Predicted Price: $17.49
